In [1]:
!pip install pandas scikit-learn transformers torch datasets
!pip install -U transformers

In [2]:
import pandas as pd

df = pd.read_csv("support_tickets.csv")
df.head()

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [3]:
df = df[["Ticket Description", "Ticket Type"]]
df = df.dropna()
df = df.sample(100, random_state=42)
df.head()

,Ticket Description,Ticket Type
4830,I'm having an issue with the {product_purchase...,Refund request
7075,I'm having trouble connecting my {product_purc...,Product inquiry
4715,I'm having an issue with the {product_purchase...,Billing inquiry
2022,I'm having an issue with the {product_purchase...,Billing inquiry
676,I'm having an issue with the {product_purchase...,Refund request


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["Ticket Description"],
    df["Ticket Type"],
    test_size=0.2,
    random_state=42
)

In [5]:
from transformers import pipeline

zero_shot = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

labels = df["Ticket Type"].unique().tolist()

C:\Users\M A D I N A\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 515/515 [00:00<00:00, 2302.95it/s]


In [6]:
def zero_shot_predict(text):
    result = zero_shot(text, labels)

    top3 = sorted(
        zip(result["labels"], result["scores"]),
        key=lambda x: x[1],
        reverse=True
    )[:3]

    return top3

In [7]:
def few_shot_prompt(text):
    return f"""
Example:
Ticket: Internet not working
Tag: Technical Issue

Example:
Ticket: Refund not received
Tag: Billing Issue

Now classify:
Ticket: {text}
Tag:
"""

In [8]:
for text in X_test.iloc[:10]:
    print("Ticket:", text)
    print("Top 3 Tags:", zero_shot_predict(text))
    print("-" * 50)

Ticket: I'm having an issue with the {product_purchased}. Please assist.

Q8. When should the new products use the brand? Ashbrook has a range of new products, but the new products aren't that common? I've already contacted customer support multiple times, but the issue remains unresolved.
Top 3 Tags: [('Product inquiry', 0.6829251646995544), ('Billing inquiry', 0.12988929450511932), ('Technical issue', 0.108836330473423)]
--------------------------------------------------
Ticket: I've encountered a data loss issue with my {product_purchased}. All the files and documents seem to have disappeared. Can you guide me on how to retrieve them? Thanks! I rely heavily on my {product_purchased} for my daily tasks, and this issue is hindering my productivity.
Top 3 Tags: [('Product inquiry', 0.6075623035430908), ('Technical issue', 0.1780497133731842), ('Billing inquiry', 0.08944873511791229)]
--------------------------------------------------
Ticket: I'm having an issue with the {product_purcha

In [9]:
from transformers import DistilBertTokenizerFast

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

In [10]:
def tokenize(batch):
    return tokenizer(batch, truncation=True, padding=True)

train_encodings = tokenizer(
    list(X_train),
    truncation=True,
    padding=True,
    max_length=128
)

test_encodings = tokenizer(
    list(X_test),
    truncation=True,
    padding=True,
    max_length=128
)

In [11]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

In [12]:
import torch

class TicketDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = TicketDataset(train_encodings, y_train_enc)
test_dataset = TicketDataset(test_encodings, y_test_enc)

In [13]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(le.classes_)
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1603.71it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=16
)

In [15]:
import numpy as np
from sklearn.metrics import accuracy_score

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {"accuracy": accuracy_score(p.label_ids, preds)}

In [16]:
from transformers import Trainer, TrainingArguments
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

C:\Users\M A D I N A\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.49s/it]


TrainOutput(global_step=10, training_loss=1.611007308959961, metrics={'train_runtime': 90.2117, 'train_samples_per_second': 1.774, 'train_steps_per_second': 0.111, 'total_flos': 3725844912000.0, 'train_loss': 1.611007308959961, 'epoch': 2.0})

In [17]:
def fine_tuned_predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    pred = outputs.logits.argmax().item()
    return le.inverse_transform([pred])[0]

In [18]:
sample = X_test.iloc[0]

print("TEXT:", sample)
print("ZERO-SHOT:", zero_shot_predict(sample))
print("FINE-TUNED:", fine_tuned_predict(sample))

TEXT: I'm having an issue with the {product_purchased}. Please assist.

Q8. When should the new products use the brand? Ashbrook has a range of new products, but the new products aren't that common? I've already contacted customer support multiple times, but the issue remains unresolved.
ZERO-SHOT: [('Product inquiry', 0.6829251646995544), ('Billing inquiry', 0.12988929450511932), ('Technical issue', 0.108836330473423)]
FINE-TUNED: Refund request


In [19]:
def top3_tags(text):
    result = zero_shot(text, labels)

    top3 = sorted(
        zip(result["labels"], result["scores"]),
        key=lambda x: x[1],
        reverse=True
    )[:3]

    return top3
df_small = df.sample(min(100, len(df)), random_state=42)

df_small["Top3 Tags"] = df_small["Ticket Description"].apply(top3_tags)

df_small.head()

,Ticket Description,Ticket Type,Top3 Tags
3684,I'm having an issue with the {product_purchase...,Cancellation request,"[(Product inquiry, 0.6829251646995544), (Billi..."
1090,I've encountered a data loss issue with my {pr...,Technical issue,"[(Product inquiry, 0.6075623035430908), (Techn..."
5141,I'm having an issue with the {product_purchase...,Product inquiry,"[(Product inquiry, 0.7162684202194214), (Techn..."
2812,I'm having an issue with the {product_purchase...,Refund request,"[(Product inquiry, 0.6093764305114746), (Techn..."
7307,I'm having an issue with the {product_purchase...,Cancellation request,"[(Product inquiry, 0.6407397985458374), (Techn..."
